# End-to-end pipeline

Runs the project modules in order. Calculations live in `src/epo_project`; this notebook only calls them and shows key outputs.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
import pandas as pd
from epo_project import config, data, replication, simulation

## Steps 1–2: industry returns, momentum and risk inputs

Downloads the French files (reused if already present), writes `data/processed` and `data/interim`, and checks the completion criteria.

In [2]:
summary = data.prepare_returns()
{k: v for k, v in summary.items() if k != "industries"}

{'first_month': '1926-07-31',
 'last_common_month': '2026-08-31',
 'missing_industry_months': 2868,
 'rf_source': 'F-F_Research_Data_Factors (monthly RF, one-month T-bill)',
 'eligibility_rule': 'momentum_ready: >= 12 consecutive months ending at t; risk_ready: >= 60; benchmark_eligible: both'}

In [3]:
data.build_inputs()

{'formation_dates': 1143,
 'first': '1931-06-30',
 'last': '2026-08-31',
 'industries_min_max': (40, 49),
 'smallest_eigenvalue_q': 3.545648451161342e-05}

## Step 3: Equity 1 benchmark validation

Raw MVO, MVO with Q and scalar EPO on the proposal grid, each scaled to 10% predicted volatility; w chosen each January from the preceding 60 months of Sharpe ratios. Published Equity 1 values are shown for comparison.

In [4]:
bench = replication.run()
bench["table"].round(3)

,portfolio,sharpe,sharpe_published,months,difference
0,Raw MVO (Sigma_hat),0.303,0.19,924,0.113
1,MVO with Q (EPO w=0),0.572,0.56,924,0.012
2,EPO w=0.25,0.720,0.75,924,-0.030
3,EPO w=0.50,0.770,0.79,924,-0.020
4,EPO w=0.75,0.796,0.80,924,-0.004
5,EPO w=0.90,0.804,0.79,924,0.014
6,EPO w=0.95,0.803,NaN,924,NaN
7,EPO w=0.99,0.794,0.73,924,0.064
8,OOS EPO,0.751,0.79,924,-0.039


In [5]:
bench["diagnostics"].round(3)

,mean_eigenvalue,realized_to_predicted_vol,ex_ante_sharpe_relative,realized_sharpe_signal_direction,risk_share_mvo_q,risk_share_oos_epo
eigenvalue_group,,,,,,
1,0.068,1.948,0.964,0.140,0.190,0.035
2,0.127,1.569,1.020,0.014,0.208,0.046
3,0.253,1.235,1.016,0.321,0.201,0.071
4,0.516,0.964,1.013,0.573,0.204,0.169
5,4.344,0.881,0.986,0.542,0.198,0.679


## Step 4: scalar compression loss

Known-covariance Gaussian model with 49 assets and 1,000 paired draws per heterogeneity level. Compares the heterogeneous allocation with the best scalar (oracle) and the mean-uncertainty scalar on the same signals; uniform `h` must give zero loss. Losses are in basis points of monthly utility.

In [ ]:
sim = simulation.run()
sim["audit"]

In [ ]:
sim["table"].round(3)

## Step 5: forecasts and relative uncertainty

At formation month t, use 120 completed pairs: predictors from t−120 through t−1 and excess-return responses from t−119 through t. Fit each industry by OLS, then refit 200 shared-date moving-block samples (12 months per block) at the fixed current signal. The variance of these fitted means is `u`; future residual variance is excluded.

Divide `u` by the trailing 60-month asset variance, winsorize at the cross-sectional 5th/95th percentiles, floor at 1e−8, and normalize to obtain mean-one `h`. Missing histories are excluded without imputation. The default universe follows the shared workflow's 120-pair requirement; the local experiment's additional 60-month admission delay is available as `minimum_ready_months=60`.

This section uses the tracked Step 1/2 CSVs and can run after the import cell without rerunning Steps 1–4. These outputs support later portfolio tests; they do not establish the proxy's calibration or economic value. Yadan can use `estimate_forecasts` directly in Step 6: see [the handoff](doc/step5_handoff.md).

In [2]:
from epo_project import reliability

reliability_results = reliability.run()
forecast_panel = reliability_results["forecasts"]
reliability_diagnostics = reliability_results["diagnostics"]
{k: v for k, v in reliability_results["manifest"].items() if k != "hashes"}

{'formation_start': '1941-06-30',
 'formation_end': '2025-11-30',
 'rows': 46662,
 'formation_months': 1014,
 'regression_pairs': 120,
 'bootstrap_draws': 200,
 'block_length': 12,
 'seed': 20261029,
 'minimum_ready_months': 1,
 'winsor_quantiles': (0.05, 0.95),
 'h_floor': 1e-08}

In [3]:
# A forecast formed in December 2018 targets January 2019.
# Training responses stop in December 2018; the target return is not an input.
example_forecasts = forecast_panel[forecast_panel["date"] == "2018-12-31"]
example_forecasts[["date", "target_date", "industry", "s", "u", "sigma2", "h",
                   "response_start", "response_end"]].head(8)

,date,target_date,industry,s,u,sigma2,h,response_start,response_end
42546,2018-12-31,2019-01-31,Agric,0.005087,0.000050,0.002708,0.817986,2009-01-31,2018-12-31
42547,2018-12-31,2019-01-31,Food,0.007588,0.000015,0.001280,0.523573,2009-01-31,2018-12-31
42548,2018-12-31,2019-01-31,Soda,0.017834,0.000030,0.001529,0.882355,2009-01-31,2018-12-31
42549,2018-12-31,2019-01-31,Beer,0.010487,0.000004,0.001174,0.306606,2009-01-31,2018-12-31
42550,2018-12-31,2019-01-31,Smoke,0.004489,0.000093,0.002936,1.416698,2009-01-31,2018-12-31
42551,2018-12-31,2019-01-31,Toys,0.008732,0.000028,0.003408,0.362901,2009-01-31,2018-12-31
42552,2018-12-31,2019-01-31,Fun,0.019391,0.000058,0.003638,0.713058,2009-01-31,2018-12-31
42553,2018-12-31,2019-01-31,Books,0.014658,0.000084,0.002552,1.467742,2009-01-31,2018-12-31


In [4]:
# CV = population standard deviation / mean of h within each formation date.
# Rank persistence is Spearman correlation with h twelve months earlier,
# restricted to common industry labels; it is unavailable for the first year.
pd.concat([reliability_diagnostics.head(3), reliability_diagnostics.tail(3)]).round(6)

,date,target_date,n_regression_ready,n_industries,h_mean,h_min,h_max,h_cv,mean_u,zero_u_count,h_rank_corr_12m
0,1941-06-30,1941-07-31,40,40,1.0,0.418412,2.488811,0.648906,0.000296,0,NaN
1,1941-07-31,1941-08-31,40,40,1.0,0.351584,2.221631,0.620889,0.000375,0,NaN
2,1941-08-31,1941-09-30,40,40,1.0,0.277874,2.611096,0.749675,0.000395,0,NaN
1011,2025-09-30,2025-10-31,49,49,1.0,0.236881,3.226092,0.846976,0.000101,0,0.324826
1012,2025-10-31,2025-11-30,49,49,1.0,0.295265,3.232051,0.799061,0.000103,0,0.173846
1013,2025-11-30,2025-12-31,49,49,1.0,0.322471,2.944952,0.808204,0.000092,0,0.144345


**Checks and interpretation.** Run `PYTHONPATH=src python -m pytest tests/test_reliability.py -q` for the numerical and timing checks. `output/reliability_manifest.json` links results to inputs, code and parameters. The core estimator also returns the full bootstrap forecast distribution for Step 6 coverage analysis.

The [alignment review](output/step5_alignment_review.md) records the missing-next-return and admission-rule decisions needed before Step 7. The subsequent 2019–2025 period has already been inspected in the local quick experiment, so it should not be described as an untouched holdout.